# SPK-2 — Executor OOM (memory pressure → GC thrash → kill)

**Break → Detect → Fix → Prove.** A job over-**caches** a wide frame, then runs a heavy
aggregation with **too few shuffle partitions**. Storage and execution memory fight over one
small heap → GC time balloons, the shuffle spills to disk, and on a tight box the executor JVM is
**OOM-killed** (`exit 137`).

**⚠️ Pre-requisite — the CONSTRAINED box:** start the server with **`make up-constrained`**
(~2 GB container, driver ~1g, 2 cores), *not* `make up`. Memory is fixed when the Spark server
boots, so a Spark Connect client **cannot** shrink the heap at runtime — only the small *box*
makes this OOM real-but-contained. On the tuned (~3 GB) box you'll still see heavy GC + spill, but
the squeeze is gentler. **Open the Spark UI at http://localhost:4040** and watch it as cells run.

**Local-mode note:** the server runs `--master local[*]`, so the **driver JVM is also the
executor** — one heap. "Executor OOM" and "driver heap exhaustion" are the same event here, the
small-scale stand-in for a killed executor on a real cluster.

**Laptop-safe:** data is *generated lazily* and only **aggregated** (never collected or written),
so nothing fills disk. Failure is contained inside the ~2 GB container; the host stays usable, and
`make clean` recovers `.tmp/`. See the companion [`README.md`](./README.md), the
[Spark-UI guide](../../docs/spark-ui-guide.md), and the [troubleshooting sheet](../../docs/troubleshooting.md).

In [1]:
from common.spark_session import spark, display_df
from common.profiles import apply_profile, profile_summary
from common.datagen import wide_rows, skewed_keys
from common.metrics_diff import measure, compare
from pyspark.sql import functions as F

# Watch this while the cells run: Executors tab (GC Time, Storage Memory) and
# Stages -> Tasks -> Summary Metrics (Spill). See docs/spark-ui-guide.md.
print("Spark UI:", "http://localhost:4040")
spark

Spark UI: http://localhost:4040


## Step 0 — Parameters & the wide dataset

We synthesize a **wide** events fact with `common.datagen.wide_rows` (~60 `double` columns per
row, so each row is fat and the frame eats memory fast — far more bytes than a narrow table at the
same row count), plus a high-cardinality `group_key` to force a heavy, memory-hungry shuffle in
the aggregation. Everything is lazy — nothing is stored until an action runs.

In [2]:
# Modest by default so the constrained box squeezes (heavy GC + spill) without certainly
# dying mid-notebook. To push it to a hard OOM (exit 137), raise N_ROWS and/or N_COLS — expect
# the container to be killed (that IS the lesson). `make clean` recovers afterwards.
N_ROWS  = 8_000_000     # try 15-20M to intensify the squeeze
N_COLS  = 60            # wide rows: ~60 doubles/row -> fat payload
N_GROUPS = 200_000      # many distinct group keys -> a big, memory-hungry shuffle

wide = wide_rows(spark, n_rows=N_ROWS, n_cols=N_COLS)
# Add a high-cardinality key to group by (deterministic, evenly spread -> not skew, just heavy).
wide = wide.withColumn("group_key", F.pmod(F.hash("row_id"), F.lit(N_GROUPS)))

# The heavy aggregation: sum every wide column per group. Lots of state to hold in execution
# memory during the shuffle. Build it once; we run it under two profiles below.
agg_cols = [F.sum(f"c{i}").alias(f"s{i}") for i in range(N_COLS)]

print(f"wide fact: ~{N_ROWS:,} rows x {N_COLS} cols (lazy)   group_key cardinality: {N_GROUPS:,}")

wide fact: ~8,000,000 rows x 60 cols (lazy)   group_key cardinality: 200,000


## 1. Break it — over-cache + too few partitions

The `constrained` session profile sets **`spark.sql.shuffle.partitions = 16`** (so each
post-shuffle partition is huge) and turns AQE off (no runtime coalescing to rescue us). Then we
make the classic mistake: **`.cache()` the whole wide frame and force it resident**, pinning a
large block of *storage* memory — right before a shuffle that needs a large block of *execution*
memory. They fight over one small heap.

In [3]:
apply_profile(spark, "constrained")    # AQE off, shuffle.partitions = 16
profile_summary(spark)

# Over-cache: pin the wide frame in storage memory. .cache() is LAZY, so .count() materializes it
# (this is also why an empty Storage tab is a common "why didn't it cache?" gotcha).
wide.cache()
pinned = wide.count()
print(f"\ncached & materialized {pinned:,} wide rows -> storage memory now pinned")

# Heavy aggregation over too-few, too-big partitions while the cache hogs the heap.
broken_agg = wide.groupBy("group_key").agg(*agg_cols)
m_broken = measure(spark, "over-cached, 16 parts", lambda: broken_agg.count())
print("\nbroken-agg metrics:", m_broken)

Applied 'constrained' session profile:
  spark.sql.adaptive.enabled                       = false
  spark.sql.adaptive.skewJoin.enabled              = false
  spark.sql.adaptive.coalescePartitions.enabled    = false
  spark.sql.autoBroadcastJoinThreshold             = -1
  spark.sql.shuffle.partitions                     = 16
Current session knobs:
  spark.sql.adaptive.enabled                       = false
  spark.sql.adaptive.skewJoin.enabled              = false
  spark.sql.adaptive.coalescePartitions.enabled    = false
  spark.sql.autoBroadcastJoinThreshold             = -1
  spark.sql.shuffle.partitions                     = 16


UnknownException: (java.lang.IllegalStateException) Cannot call methods on a stopped SparkContext.
This stopped SparkContext was created at:

org.apache.spark.sql.hive.thriftserver.HiveThriftServer2.main(HiveThriftServer2.scala)
java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:77)
java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:43)
java.base/java.lang.reflect.Method.invoke(Method.java:569)
org.apache.spark.deploy.JavaMainApplication.start(SparkApplication.scala:52)
org.apache.spark.deploy.SparkSubmit.org$apache$spark$deploy$SparkSubmit$$runMain(SparkSubmit.scala:1027)
org.apache.spark.deploy.SparkSubmit.doRunMain$1(SparkSubmit.scala:204)
org.apache.spark.deploy.SparkSubmit.submit(SparkSubmit.scala:227)
org.apache.spark.deploy.SparkSubmit.doSubmit(SparkSubmit.scala:96)
org.apache.spark.deploy.SparkSubmit$$anon$2.d

## 2. Detect it — read the Spark UI

http://localhost:4040 → **Executors** tab. The leading indicators of an executor OOM
(see [`docs/spark-ui-guide.md`](../../docs/spark-ui-guide.md)):

- **Task Time (GC Time):** the parenthesized GC figure is a **large fraction** of task time → the
  JVM is spending its time collecting garbage instead of working — the precursor to the OOM.
- **Storage Memory:** pinned **near the limit** — our `.cache()` is crowding out execution memory.
- **stderr** (linked per executor) or the driver log: `OutOfMemoryError` /
  **`GC overhead limit exceeded`**; an executor that **vanishes**; process **`exit 137`** (the OS
  killed the JVM for exceeding the ~2 GB container).

Then **Stages → Tasks → Summary Metrics**: **Spill (memory)** and **Spill (disk)** go **non-zero**
— data didn't fit execution memory. The cell below prints the spill / peak-memory numbers
`metrics_diff` could read over Spark Connect (GC time itself is a JVM-internal metric you read in
the Executors tab, not over the Connect REST surface).

In [4]:
def show_memory(m, title):
    print(f"== {title} ==")
    print(f"  tasks            : {m['num_tasks']}")
    print(f"  spill (memory)   : {m['spill_mem_bytes']} bytes")
    print(f"  spill (disk)     : {m['spill_disk_bytes']} bytes   <- data overflowed execution memory")
    print(f"  peak exec memory : {m['peak_mem_max_bytes']} bytes")
    print(f"  runtime          : {m['runtime_s']} s")

show_memory(m_broken, "BROKEN (over-cached, 16 partitions)")
print("\nNow look at the Executors tab: GC Time as a big fraction of Task Time, and Storage")
print("Memory pinned near the limit. On the constrained box, watch stderr for exit 137.")

NameError: name 'm_broken' is not defined

## 3. Diagnose

A task must hold its **whole partition** in execution memory to process it. With
`shuffle.partitions = 16`, each partition is enormous → per-task memory is enormous. Meanwhile the
`.cache()` pinned a big slice of the *same* heap as **storage**, leaving less for execution.
Spark's unified memory manager lets the two share a pool (governed by `spark.memory.fraction`), but
when both want a lot at once on a small heap it **spills** to disk, **GC** churns trying to free
space, and if it still can't keep up the JVM throws `OutOfMemoryError` — or the cgroup kills the
container (`exit 137`). More cores wouldn't help: the limit is **bytes per partition**, not
parallelism.

## 4. Fix it — more partitions + stop over-caching

Two levers, applied together:

1. **`unpersist()` the needless cache.** We scan `wide` once here — caching it is pure cost. Drop
   it and hand that memory back to execution. (Cache only what you reuse *many* times and that
   *fits*.)
2. **Raise `shuffle.partitions`.** The `tuned` profile sets 200 (and AQE on, so it can coalesce).
   More, smaller partitions → each task holds far less at once → execution memory fits and spill
   drops. **This is the biggest lever.**

*(A third lever, `spark.memory.fraction`, is the JVM-level storage/execution split — but it's
fixed at server startup, so a Connect client can't change it here. On a real cluster you'd size it
along with `executor.memory`; we just name it.)*

In [ ]:
# Fix 1: release the cache we don't reuse -> storage memory returns to execution.
wide.unpersist(blocking=True)

# Fix 2: more, smaller partitions (+ AQE on to coalesce). Let Spark spill transiently if needed
# rather than pinning the input.
apply_profile(spark, "tuned")          # shuffle.partitions = 200, AQE on
profile_summary(spark)

fixed_agg = wide.groupBy("group_key").agg(*agg_cols)
m_fixed = measure(spark, "unpersist, 200 parts", lambda: fixed_agg.count())
show_memory(m_fixed, "FIXED (no cache, 200 partitions)")

## 5. Prove it

Before/after. Watch **Spill (memory)** and **Spill (disk)** collapse toward zero and **Peak exec
memory** drop as partitions get smaller and the cache stops hogging the heap — corroborated in the
**Executors** tab by GC Time and Storage Memory falling back to a healthy range.

In [ ]:
compare([m_broken, m_fixed])

## Takeaways & "in real production…"

- **Detect early:** **GC Time as a fraction of Task Time** climbing + **Storage Memory pinned at
  the limit** + **non-zero spill** — *before* the executor dies. `exit 137` / `container killed` is
  the late symptom; GC/spill pressure is the warning.
- **Too few partitions is the usual culprit:** per-partition **bytes**, not core count, is the
  constraint. Raise `shuffle.partitions` / keep **AQE coalesce** on so partitions fit the heap.
- **Cache deliberately:** only what you reuse many times *and* that fits; always `.unpersist()`
  when done. Prefer letting Spark **spill** a shuffle (cheap, transient) over **pinning** an input
  you scan once (expensive, resident). A forgotten cache is a classic slow-burn OOM.
- **The "shrink the box" trick:** memory is fixed at server boot, so this module needs
  `make up-constrained` — that turns a gentle laptop squeeze into a real-but-contained failure. On
  real clusters you'd size `executor.memory` / `spark.memory.fraction` instead.
- **In prod:** alert on executor GC-time ratio and OOM kills / `FetchFailedException` (a vanished
  executor's shuffle blocks go missing downstream); keep shuffle partitions in the
  tens-to-hundreds-of-MB range; review every `.cache()` in code review.

## Teardown

Nothing was written (we only aggregated generated data), so there is nothing to delete. We release
any remaining cache and restore the production-tuned safety nets. If the constrained container was
OOM-killed mid-run, `make up-constrained` restarts it; `make clean` clears anything left in `.tmp/`.

In [ ]:
apply_profile(spark, "tuned")        # restore production-tuned safety nets
spark.catalog.clearCache()           # drop any persisted frames
print("Done. Profile reset to 'tuned'; caches cleared. No tables/files were created;")
print("`make clean` clears .tmp if needed.")